# NB09 · Evaluación consolidada, atribución de errores y artefactos

Objetivo: la tabla que decide, los tres artefactos de entrega, y la atribución de ≥3 fallos. No hay decisiones D en este notebook -las 22 ya están cerradas-; lo que hace falta es reunir lo que NB01-NB08 ya midieron, generar lo único que falta (el top-10 de las 12 consultas de evaluación) y razonar sobre los fallos con evidencia real.

### 🗺️ Mapa de datos: de dónde sale cada número de la tabla comparativa

Ningún notebook anterior dejó la fila del ANN elegido en un artefacto -NB06 solo la mostró en pantalla-, así que es la única fila que este notebook **recalcula** en vez de leer. Todo lo demás es lectura de artefactos ya escritos.

| Fila de la tabla | Viene de | Cómo |
|---|---|---|
| TF-IDF, BM25 | `artifacts/baseline_lexico.json` (NB01) | lectura directa |
| Denso, oráculo exacto (R01, plantilla A4) | `artifacts/comparativa_representacion.json` (NB03) | lectura directa, fila `posicion_regla == 1` |
| Denso, modelo ganador sobre la muestra (R02) | `artifacts/comparativa_modelos.json` (NB02) | lectura directa -⚠️ sobre `catalogo_muestra`, no comparable en crudo con el resto- |
| **Denso, ANN elegido (R04, `ef=32`)** | recalculado aquí (sección B), latencia de `artifacts/benchmark_ann.csv` | única fila que no viene de un JSON ya escrito |

El resto del notebook (`resultados_busqueda.csv`, consistencia entre formulaciones, consultas filtradas, atribución de errores) no depende de esta tabla: son piezas independientes que comparten la misma conexión a Qdrant.

In [1]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000, desde cache) + consultas_desarrollo.csv (8) + consultas_evaluacion.csv (12) + consultas_filtradas.csv (4)
import json
import os
import sys
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from dotenv import load_dotenv

from aurum.ann import comparar_ndcg_con_oraculo
from aurum.busqueda import BuscadorVectorial, DenseRetriever, auditar_filtro_de_marca, rank_queries_dense
from aurum.consolidacion import diagnosticar_consulta, fila_comparativa, tabla_comparativa
from aurum.datos import load_csv
from aurum.embeddings import (
    GeminiEncoder, cache_key, corpus_fingerprint, encode_corpus, truncate_dim,
)
from aurum.evaluacion import formulation_consistency, qrels_from_judgements
from aurum.motores import CATALOG_PREFIX, catalog_collection_name
from aurum.motores.qdrant import QdrantStore
from aurum.plantillas import render_template

load_dotenv(Path("..") / ".env")
DATA = Path("..") / "data"
CACHE = Path("..") / "artifacts" / "embeddings"
completo = load_csv(DATA / "catalogo_productos.csv")
desarrollo = load_csv(DATA / "consultas_desarrollo.csv")
evaluacion = load_csv(DATA / "consultas_evaluacion.csv")
relevancias = load_csv(DATA / "relevancias_desarrollo.csv")
filtradas = load_csv(DATA / "consultas_filtradas.csv")
QRELS = qrels_from_judgements(relevancias)

MODELO, CONTRATO, PLANTILLA = "gemini-embedding-2", "sin_contrato", "A4"
DIM, TOP_K = 768, 10
COLECCION = catalog_collection_name(model=MODELO, template=PLANTILLA, dim=DIM)
EF_ELEGIDO = 32   # R04 (NB06)

print(f"coleccion : {COLECCION} · ef={EF_ELEGIDO}")
print(f"consultas : {len(desarrollo)} desarrollo · {len(evaluacion)} evaluacion · {len(filtradas)} filtradas")

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


coleccion : aurum_catalogo__gemini_embedding_2__A4__768 · ef=32
consultas : 8 desarrollo · 12 evaluacion · 4 filtradas


## A · Conexión, oráculo exacto y buscador ANN

**Entrada:** las constantes del setup. **Salida:** `oraculo` (`DenseRetriever` sobre los 15.000, exacto -igual que en NB02/NB06-) y `buscador` (`BuscadorVectorial` contra Qdrant con `ef=32`, el mismo R04 que ya usaron NB07 y NB08). Los vectores del catálogo salen de la cache de NB04 -si no están, la celda para en vez de pagar 15.000 llamadas-.

In [2]:
# ⚠️ requiere `make motor-up MOTOR=qdrant`
CORPUS_ID = f"catalogo_productos__{PLANTILLA}"
textos_completo = render_template(completo, PLANTILLA)
clave = cache_key(
    model_id=MODELO, kind="document", contract=CONTRATO,
    corpus_id=CORPUS_ID, fingerprint=corpus_fingerprint(textos_completo),
)
if not (CACHE / f"{clave}.npy").exists():
    raise RuntimeError(
        f"Los vectores de {CORPUS_ID} no estan en cache ({clave}).\n"
        f"Deberian estar desde NB04/NB06 -esta celda no paga 15.000 llamadas nuevas."
    )

_encoder = GeminiEncoder(
    api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
    native_dim=3072, window=8192,
)
codificado_completo = encode_corpus(
    _encoder, textos_completo, corpus_id=CORPUS_ID,
    kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
)
vectores_completo = truncate_dim(codificado_completo.vectors, DIM)
ids_completo = completo["product_id"].tolist()
oraculo_retriever = DenseRetriever(vectores_completo, ids_completo, metric="cosine")


@lru_cache(maxsize=256)
def codificar_consulta(texto: str):
    codificado = encode_corpus(
        _encoder, [texto], corpus_id="consulta_suelta",
        kind="query", contract=CONTRATO, batch_size=1, cache_dir=CACHE,
    )
    return truncate_dim(codificado.vectors, DIM)[0]


almacen = QdrantStore(
    collection=COLECCION,
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    prefix=CATALOG_PREFIX,
    timeout=30,
)
buscador = BuscadorVectorial(almacen, codificar_consulta, top_k=TOP_K, ef=EF_ELEGIDO)
print(f"puntos en la coleccion: {almacen.count():,}".replace(",", ".") +
      f" · indice al dia: {almacen.index_ready()}")

puntos en la coleccion: 15.000 · indice al dia: True


## B · La fila que falta: ANN elegido (R04) sobre las 8 de desarrollo

**Entrada:** `oraculo_retriever`, `buscador`, `desarrollo`, `QRELS`. **Salida:** `metricas_ann_elegido` -un dict con `ndcg_at_10`/`recall_at_10`/`mrr_at_10`-, usado en la sección C. Mismo cálculo que la sección G de `06_ann.ipynb`, reejecutado porque esa tabla nunca se guardó en un artefacto.

In [3]:
QUERY_IDS_DEV = [str(q) for q in desarrollo["query_id"]]
vectores_dev = encode_corpus(
    _encoder, desarrollo["query_text"].tolist(), corpus_id="consultas_desarrollo",
    kind="query", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
).vectors
vectores_dev = truncate_dim(vectores_dev, DIM)

oraculo_dev = rank_queries_dense(oraculo_retriever, QUERY_IDS_DEV, vectores_dev, k=TOP_K)
ann_dev = {
    qid: [r.document_id for r in buscador.buscar(texto, top_k=TOP_K)]
    for qid, texto in zip(QUERY_IDS_DEV, desarrollo["query_text"])
}

tabla_ndcg = comparar_ndcg_con_oraculo(ann_dev, oraculo_dev, QRELS, k=TOP_K)
metricas_ann_elegido = (
    tabla_ndcg[tabla_ndcg["sistema"] == "ANN elegido (R04)"].iloc[0].to_dict()
)
tabla_ndcg.style.hide(axis="index").format({
    columna: "{:.1%}" for columna in tabla_ndcg.columns if columna != "sistema"
})

sistema,precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10
oráculo exacto (DenseRetriever),62.5%,29.0%,93.8%,60.1%
ANN elegido (R04),57.5%,27.3%,56.2%,47.9%


## C · La tabla comparativa

**Entrada:** los tres artefactos ya escritos (mapa de datos, arriba) + `metricas_ann_elegido` de B + la fila `ef=32` de `benchmark_ann.csv` para la latencia. **Salida:** `tabla_comparativa_final`, escrita como `artifacts/tabla_comparativa.md` en la sección H.

La fila del modelo ganador (R02) queda marcada como no comparable en crudo -se decidió sobre `catalogo_muestra` (1.500), no sobre el catálogo completo- para no enseñar cuatro números en la misma columna que en realidad miden corpus distintos.

In [4]:
baseline = json.loads((Path("..") / "artifacts" / "baseline_lexico.json").read_text(encoding="utf-8"))
representacion = json.loads((Path("..") / "artifacts" / "comparativa_representacion.json").read_text(encoding="utf-8"))
modelos = json.loads((Path("..") / "artifacts" / "comparativa_modelos.json").read_text(encoding="utf-8"))
benchmark_ann = pd.read_csv(Path("..") / "artifacts" / "benchmark_ann.csv")

def _ganadora(regla, nombre):
    # Misma exigencia que consolidar_metricas._ganadora: la posicion 1
    # sin "admisible" no es una ganadora, es la mejor de un barrido sin
    # ganador -devolverla igual escondería que la regla no eligió nada-.
    primera = next(f for f in regla if f["posicion_regla"] == 1)
    if not primera.get("admisible"):
        raise RuntimeError(f"{nombre} no dejo ninguna configuracion admisible.")
    return primera


ganadora_r01 = _ganadora(representacion["regla_r01_completo"], "R01")
ganadora_r02 = _ganadora(modelos["regla_d09b"], "R02")
fila_ann = benchmark_ann[benchmark_ann["ef"] == EF_ELEGIDO].iloc[0]

filas = [
    fila_comparativa(
        "C0a · TF-IDF", modelo="TF-IDF", metrica="coseno TF-IDF", ann="exacto",
        **{k: baseline["completo"]["metricas"]["tfidf"][k] for k in
           ("ndcg_at_10", "recall_at_10", "mrr_at_10")},
    ),
    fila_comparativa(
        "C0b · BM25", modelo="BM25", metrica="BM25", ann="exacto",
        **{k: baseline["completo"]["metricas"]["bm25"][k] for k in
           ("ndcg_at_10", "recall_at_10", "mrr_at_10")},
    ),
    fila_comparativa(
        "C1 · denso, muestra (R02)", modelo=ganadora_r02["modelo"], dim=ganadora_r02["dim"],
        metrica="coseno", ann="exacto",
        ndcg_at_10=ganadora_r02["ndcg_at_10"], recall_at_10=ganadora_r02["recall_at_10"],
        mrr_at_10=ganadora_r02["mrr_at_10"],
        nota="sobre catalogo_muestra (1.500) -no comparable en crudo con el resto, sobre 15.000-",
    ),
    fila_comparativa(
        "C2 · denso, oraculo exacto (R01)", modelo=MODELO, plantilla=ganadora_r01["plantilla"],
        dim=DIM, metrica="coseno", ann="exacto",
        ndcg_at_10=ganadora_r01["ndcg_at_10"], recall_at_10=ganadora_r01["recall_at_10"],
        mrr_at_10=ganadora_r01["mrr_at_10"],
    ),
    fila_comparativa(
        "C3 · denso, ANN elegido (R04) — el sistema real", modelo=MODELO, plantilla=PLANTILLA,
        dim=DIM, metrica="coseno", ann=f"Qdrant HNSW ef={EF_ELEGIDO}",
        ndcg_at_10=metricas_ann_elegido["ndcg_at_10"],
        recall_at_10=metricas_ann_elegido["recall_at_10"],
        mrr_at_10=metricas_ann_elegido["mrr_at_10"],
        p50_ms=float(fila_ann["ms_p50"]), p95_ms=float(fila_ann["ms_p95"]),
        nota="la distancia con C2 se lee con la nota al pie -el oraculo de C2 y este indice no comparan el mismo corpus, ver G.1.c-",
    ),
]
tabla_comparativa_final = tabla_comparativa(filas)
tabla_comparativa_final.style.hide(axis="index")

config,modelo,plantilla,dim,metrica,ann,ndcg_at_10,recall_at_10,mrr_at_10,p50_ms,p95_ms,nota
C0a · TF-IDF,TF-IDF,nan,nan,coseno TF-IDF,exacto,0.412900,0.151000,0.750000,nan,nan,nan
C0b · BM25,BM25,nan,nan,BM25,exacto,0.508800,0.184100,0.750000,nan,nan,nan
"C1 · denso, muestra (R02)",gemini-2,nan,768.000000,coseno,exacto,0.771800,0.415500,1.000000,nan,nan,"sobre catalogo_muestra (1.500) -no comparable en crudo con el resto, sobre 15.000-"
"C2 · denso, oraculo exacto (R01)",gemini-embedding-2,A4,768.000000,coseno,exacto,0.600600,0.290500,0.937500,nan,nan,nan
"C3 · denso, ANN elegido (R04) — el sistema real",gemini-embedding-2,A4,768.000000,coseno,Qdrant HNSW ef=32,0.478700,0.272500,0.562500,8.510000,11.100000,"la distancia con C2 se lee con la nota al pie -el oraculo de C2 y este indice no comparan el mismo corpus, ver G.1.c-"


## D · `resultados_busqueda.csv` — top-10 de las 12 consultas de evaluación

**Entrada:** `evaluacion` (setup) + `buscador` (A) — la primera vez que este notebook escribe un CSV de entrega, no solo lee. **Salida:** `resultados/resultados_busqueda.csv`, 120 filas (12 × 10).

Van con el `buscador` real -el mismo `ef=32` de producción, no el oráculo-, porque es lo que el sistema entregaría de verdad ante estas 12 consultas.

In [5]:
filas_resultados = [
    {
        "evaluation_id": evaluation_id,
        "rank": resultado.rank,
        "product_id": resultado.document_id,
        "score": resultado.score,
    }
    for evaluation_id, texto in zip(evaluacion["evaluation_id"], evaluacion["query_text"])
    for resultado in buscador.buscar(texto, top_k=TOP_K)
]
resultados_busqueda = pd.DataFrame(filas_resultados)

destino_busqueda = Path("..") / "resultados" / "resultados_busqueda.csv"
resultados_busqueda.to_csv(destino_busqueda, index=False)
filas_por_consulta = resultados_busqueda.groupby("evaluation_id").size()
print(f"Escrito {destino_busqueda} · {len(resultados_busqueda)} filas\n"
      f"consultas con menos de {TOP_K} resultados: "
      f"{(filas_por_consulta < TOP_K).sum()}/{len(filas_por_consulta)}\n"
      f"product_id repetido dentro de alguna consulta: "
      f"{(resultados_busqueda.groupby('evaluation_id')['product_id'].nunique() < filas_por_consulta).sum()}")
resultados_busqueda.head(10).style.hide(axis="index")

Escrito ..\resultados\resultados_busqueda.csv · 120 filas
consultas con menos de 10 resultados: 0/12
product_id repetido dentro de alguna consulta: 0


evaluation_id,rank,product_id,score
EVAL-100455-context,1,AURUM-NEW-001,0.685427
EVAL-100455-context,2,B07JHCZ1T4,0.643117
EVAL-100455-context,3,B09BQTS4FF,0.638992
EVAL-100455-context,4,B09CYWTYVD,0.629157
EVAL-100455-context,5,B07MJKBGYC,0.611683
EVAL-100455-context,6,B07C2TM76Y,0.610222
EVAL-100455-context,7,B07ZCKTM2R,0.609863
EVAL-100455-context,8,B01A5VQHBY,0.603379
EVAL-100455-context,9,B075FPNWM2,0.602323
EVAL-100455-context,10,B07XD4Q3ZV,0.600347


## E · Consistencia entre formulaciones (Jaccard, sin etiquetas)

**Entrada:** los mismos rankings de D, agrupados por intención. **Salida:** `tabla_consistencia`, 4 filas -una por intención-, al artefacto en H.

Sin juicios para las 12 de evaluación, esta es la única evidencia de calidad que no depende de etiquetas: si `direct`/`context`/`semantic` de la misma intención devuelven catálogos parecidos, el sistema entiende la intención y no solo la superficie léxica.

In [6]:
rankings_evaluacion = {
    evaluation_id: grupo.sort_values("rank")["product_id"].tolist()
    for evaluation_id, grupo in resultados_busqueda.groupby("evaluation_id")
}
tabla_consistencia = formulation_consistency(rankings_evaluacion, k=TOP_K)
tabla_consistencia.style.hide(axis="index")

intencion,jaccard_context_direct,jaccard_context_semantic,jaccard_direct_semantic
100455,0.666700,0.333300,0.250000
101352,0.538500,0.250000,0.333300
93437,0.176500,0.250000,0.052600
96202,0.333300,0.666700,0.333300


## F · Consultas filtradas: pureza y cobertura

**Entrada:** `filtradas` (setup) + `buscador`. **Salida:** `tabla_filtros`, al artefacto en H.

`auditar_filtro_de_marca` (NB05) ya resuelve esto: compara contra `alcance` -cuántos productos de esa marca hay realmente en el catálogo- para distinguir un cero legítimo de un filtro roto, y una cobertura corta de una marca con pocos productos.

In [7]:
alcance_por_marca = completo["brand"].value_counts().to_dict()
tabla_filtros = auditar_filtro_de_marca(
    buscador, filtradas.to_dict("records"), alcance=alcance_por_marca, top_k=TOP_K,
)
# "veredicto" ya distingue una respuesta vacia legitima (la marca no
# tiene productos) de un filtro roto (la marca SI tiene y devolvio 0):
# comparar solo de_la_marca == n_resultados confundiria las dos.
FILTROS_OK = int(tabla_filtros["veredicto"].str.startswith("✅").sum())
print(f"consultas filtradas puras: {FILTROS_OK}/{len(tabla_filtros)}")
tabla_filtros.style.hide(axis="index")

consultas filtradas puras: 4/4


caso,consulta,marca,n_en_catalogo,n_resultados,de_la_marca,pureza,veredicto
FILTER-001,herramienta inalámbrica para perforar,Einhell,30,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-002,tableta ligera para estudiar y tomar apuntes,Apple,100,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-003,zapatillas cómodas para salir a correr,NIKE,295,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-004,monitor para trabajar con varias ventanas,SAMSUNG,155,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)


## G · Atribución de ≥3 fallos

Procedimiento, en orden: (1) ¿ya es malo en el oráculo exacto? → **representación**; (2) ¿el oráculo lo recupera y el ANN no? → **índice**; (3) ¿falta el producto o lo excluye el filtro? → **datos/filtros**; (4) ¿el estado leído no coincide con la traza de NB08? → **persistencia**.

**Entrada:** `oraculo_dev`/`ann_dev` (B), `QRELS` (setup), `tabla_consistencia` (E). **Salida:** evidencia impresa -no un veredicto-. Las celdas de abajo solo *reúnen* la evidencia de las capas 1 y 2 para las dos candidatas de desarrollo señaladas de antemano (`diagnosticar_consulta`, `src/aurum/consolidacion.py`) y la divergencia Jaccard de la peor intención entre formulaciones; la conclusión -qué capa falló y por qué, con el `product_id` que lo sostiene- se redacta en la celda de markdown de después, mirando estos números. Falta a propósito: precodificarla sin haber visto los datos sería inventar el hallazgo.

In [8]:
def _evidencia_capas_1_2(query_id_int):
    qid = str(query_id_int)
    texto = desarrollo.loc[desarrollo["query_id"] == query_id_int, "query_text"].iloc[0]
    diagnostico = diagnosticar_consulta(
        qid, ranking_oraculo=oraculo_dev[qid], ranking_ann=ann_dev[qid],
        qrels=QRELS[qid], k=TOP_K,
    )
    print(f"consulta {qid} \"{texto}\"")
    print(f"  relevantes en el oraculo (top-{TOP_K}): "
          f"{diagnostico['n_relevantes_en_oraculo']} -> {diagnostico['relevantes_en_oraculo']}")
    print(f"  relevantes en el ANN (top-{TOP_K})    : "
          f"{diagnostico['n_relevantes_en_ann']} -> {diagnostico['relevantes_en_ann']}")
    print(f"  relevantes que el ANN perdio          : {diagnostico['perdidos_por_el_ann']}")
    return diagnostico


diagnostico_13357 = _evidencia_capas_1_2(13357)   # "base tapizada 160x200 sin patas"
print()
diagnostico_33633 = _evidencia_capas_1_2(33633)   # "disfraz halloween talla grande hombre"

consulta 13357 "base tapizada 160x200 sin patas"
  relevantes en el oraculo (top-10): 7 -> ['B01MV80F5C', 'B015FMZ4MQ', 'B07S946NHS', 'B09BW133YW', 'B00YMT0IKW', 'B08JM5M345', 'B00YMSZDZS']
  relevantes en el ANN (top-10)    : 6 -> ['B01MV80F5C', 'B015FMZ4MQ', 'B07S946NHS', 'B09BW133YW', 'B00YMT0IKW', 'B08JM5M345']
  relevantes que el ANN perdio          : ['B00YMSZDZS']

consulta 33633 "disfraz halloween talla grande hombre"
  relevantes en el oraculo (top-10): 2 -> ['B07XD9D2HB', 'B07TFKMTJ8']
  relevantes en el ANN (top-10)    : 3 -> ['B07XD9D2HB', 'B07TFKMTJ8', 'B07XM84G9L']
  relevantes que el ANN perdio          : []


### G.1 · Capa 3 (datos/filtros), solo si hace falta

Si `perdidos_por_el_ann` de alguna de las dos de arriba está vacío pero `n_relevantes_en_oraculo` también es bajo, antes de concluir "representación" hay que comprobar si el producto relevante ni siquiera está indexado o le falta el metadato que necesitaría un filtro -`indice.get(record_id)` y mirar su payload, mismo patrón que NB08 sección D-.

### G.1.b · PRUEBAS POST-RESULTADOS · NO DECISIONES — ¿es `ef` la palanca?

**Problema.** Cinco relevantes que el oráculo trae y el ANN (`ef=32`) no. Si fuera pérdida del índice, aflojar la aproximación debería recuperarlos.

**Se comprueba.** El mismo top-10 con `ef` ∈ {32, 64, 128, 256} sobre las 8 de desarrollo. Buscador aparte: no toca `buscador`, `ann_dev` ni ningún artefacto — B-F siguen siendo `ef=32`, y R04 no se reabre (la latencia de cada `ef` está en `benchmark_ann.csv`).

**Lectura.** `relevantes_top10_*`: más alto es mejor, el oráculo es el techo. `perdidos_*`: más corto es mejor, `-` = ninguno. Se lee en horizontal, por fila.

In [9]:
# PRUEBAS POST-RESULTADOS - NO DECISIONES: R04 (ef=32) no se reabre aqui.
# Buscadores aparte a proposito -no reasignan `buscador` ni `ann_dev`-.
EFS_PRUEBA = (32, 64, 128, 256)

ann_por_ef = {
    ef: {
        qid: [
            r.document_id
            for r in BuscadorVectorial(
                almacen, codificar_consulta, top_k=TOP_K, ef=ef,
            ).buscar(texto, top_k=TOP_K)
        ]
        for qid, texto in zip(QUERY_IDS_DEV, desarrollo["query_text"])
    }
    for ef in EFS_PRUEBA
}


def _fila_barrido(query_id_int):
    qid = str(query_id_int)
    fila = {"consulta": qid}
    for ef in EFS_PRUEBA:
        d = diagnosticar_consulta(
            qid, ranking_oraculo=oraculo_dev[qid], ranking_ann=ann_por_ef[ef][qid],
            qrels=QRELS[qid], k=TOP_K,
        )
        fila["relevantes_top10_oraculo_exacto"] = d["n_relevantes_en_oraculo"]
        fila[f"relevantes_top10_ann_ef{ef}"] = d["n_relevantes_en_ann"]
        fila[f"perdidos_ef{ef}"] = ", ".join(d["perdidos_por_el_ann"]) or "-"
    return fila


# Las 8 de desarrollo, no solo la 13357: una consulta suelta no distingue
# "subir ef arregla este fallo" de "subir ef mueve resultados en todas partes".
tabla_verificacion_ef = pd.DataFrame(
    [_fila_barrido(q) for q in desarrollo["query_id"]]
)
ann_verificacion = ann_por_ef[64]   # lo usa G.1.c
tabla_verificacion_ef.style.hide(axis="index")

consulta,relevantes_top10_oraculo_exacto,relevantes_top10_ann_ef32,perdidos_ef32,relevantes_top10_ann_ef64,perdidos_ef64,relevantes_top10_ann_ef128,perdidos_ef128,relevantes_top10_ann_ef256,perdidos_ef256
13357,7,6,B00YMSZDZS,6,B00YMSZDZS,6,B00YMSZDZS,6,B00YMSZDZS
18868,3,0,"B07H97VGBP, B07H2Y8R6Y, B07GN7XRQ9",0,"B07H97VGBP, B07H2Y8R6Y, B07GN7XRQ9",3,-,3,-
28703,7,7,-,7,-,7,-,7,-
31224,4,4,-,4,-,4,-,4,-
33633,2,3,-,2,-,2,-,2,-
38249,7,7,-,7,-,7,-,7,-
43240,10,9,B08MQ42Z6P,9,B08MQ42Z6P,9,B08MQ42Z6P,9,B08MQ42Z6P
61533,10,10,-,10,-,10,-,10,-


### G.1.c · PRUEBAS POST-RESULTADOS · NO DECISIONES — ¿qué pierde de verdad el ANN?

**Problema.** El oráculo se construye desde `catalogo_productos.csv` (foto previa a NB08) y el índice lleva los 24 eventos aplicados. Ambos tienen 15.000 puntos, pero no los mismos: 8 bajas y 8 altas. Y `AURUM-NEW-008` se titula *"Base tapizada 160 x 200 sin patas"* — literalmente la consulta 13357. Si el ANN lo coloca en su top-10, desplaza a un relevante **sin haberse equivocado**: conoce un producto que el oráculo no puede ver. Con ese sesgo dentro, `perdidos_por_el_ann` no mide pérdida del ANN.

**Se comprueba.** Un oráculo exacto sobre el mismo corpus que el índice, sin coste de API: los 16 vectores de los `UPSERT` están cacheados desde NB08. Se replica la semántica de Qdrant, que indexa por `record_id`: −8 bajas, 8 reescrituras, +8 altas = 15.000, que debe cuadrar con `almacen.count()`.

⚠️ NB08 codificó los upserts con `text` **crudo**, sin pasar por A4. Para las 8 altas da igual (66-96 caracteres, bajo el corte de 936); **4 de las 8 actualizaciones sí lo superan** —hasta 2.676— y quedaron indexadas sin recortar, a diferencia de los otros 14.996 puntos. El oráculo corregido lo reproduce a propósito: replica lo que el índice contiene, no lo que debería contener.

**Lectura.** `lo_pierde_vs_oraculo_post_nb08 = False` es una pérdida que nunca existió. `puesto_en_oraculo_post_nb08` = 11 significa efecto de frontera, no fallo de recuperación.

⚠️ `puesto_en_ann_ef_efectivo_200` **no** es la búsqueda de producción: Qdrant exige `hnsw_ef >= limit`, así que pedir 200 resultados sube el `ef` efectivo a 200. La columna dice dónde cae el producto en una búsqueda casi exacta — sirve para separar "el ANN lo entierra" de "el ANN ni lo encuentra", no para leer el comportamiento con `ef=32`.

In [10]:
# PRUEBAS POST-RESULTADOS - NO DECISIONES. Oraculo sobre el corpus POST-NB08,
# para que oraculo y ANN comparen por fin el mismo catalogo.
from aurum.mutaciones import clasificar_eventos
from aurum.plantillas import corpus_context

eventos = clasificar_eventos(load_csv(DATA / "eventos_catalogo.csv"))
eventos_upsert = eventos[eventos["tipo"] != "baja"].reset_index(drop=True)
eventos_baja = eventos[eventos["tipo"] == "baja"].reset_index(drop=True)

# Mismo corpus_id, mismo texto y mismo orden que NB08 -> la huella casa y
# sale de cache. Si no casara, serian 16 llamadas de pago: se comprueba.
textos_upsert = eventos_upsert["text"].tolist()
clave_upsert = cache_key(
    model_id=MODELO, kind="document", contract=CONTRATO,
    corpus_id="eventos_catalogo_upsert", fingerprint=corpus_fingerprint(textos_upsert),
)
if not (CACHE / f"{clave_upsert}.npy").exists():
    raise RuntimeError(
        f"Los vectores de eventos_catalogo_upsert no estan en cache ({clave_upsert}).\n"
        f"Deberian estar desde NB08 -esta celda no paga llamadas nuevas."
    )
vectores_upsert = truncate_dim(
    encode_corpus(
        _encoder, textos_upsert, corpus_id="eventos_catalogo_upsert",
        kind="document", contract=CONTRATO, batch_size=16, cache_dir=CACHE,
    ).vectors,
    DIM,
)

# Qdrant indexa por record_id: una baja lo borra y un upsert lo reescribe.
# Se conserva de la base todo lo que ningun evento toca, y se anaden los 16.
tocados = set(eventos_baja["record_id"]) | set(eventos_upsert["record_id"])
conserva = ~completo["record_id"].isin(tocados)
vectores_post = np.vstack([vectores_completo[conserva.to_numpy()], vectores_upsert])
ids_post = (
    completo.loc[conserva, "product_id"].tolist()
    + eventos_upsert["product_id"].tolist()
)
oraculo_post = DenseRetriever(vectores_post, ids_post, metric="cosine")
oraculo_dev_post = rank_queries_dense(oraculo_post, QUERY_IDS_DEV, vectores_dev, k=TOP_K)

largos_upsert = eventos_upsert["text"].fillna("").str.len()
CORTE_A4 = corpus_context(completo).a4_chars
print(f"productos en el oraculo corregido : {len(ids_post)}")
print(f"puntos en el indice Qdrant        : {almacen.count()}")
print(f"  reparto: -{len(eventos_baja)} bajas · "
      f"{int((eventos['tipo'] == 'actualizacion').sum())} reescritas · "
      f"+{int((eventos['tipo'] == 'alta').sum())} altas")
print(f"upserts codificados sin A4 que superan el corte de {CORTE_A4}: "
      f"{int((largos_upsert > CORTE_A4).sum())} de {len(eventos_upsert)}")

productos en el oraculo corregido : 15000
puntos en el indice Qdrant        : 15000
  reparto: -8 bajas · 8 reescritas · +8 altas
upserts codificados sin A4 que superan el corte de 936: 4 de 16


In [11]:
# PRUEBAS POST-RESULTADOS - NO DECISIONES: sigue sin reabrirse R04.
PROFUNDIDAD = 200
# OJO: Qdrant exige hnsw_ef >= limit, asi que pedir 200 resultados sube el
# ef efectivo a 200. Esto NO es el buscador de produccion: es una busqueda
# casi exacta, util solo para saber si el producto esta o no esta.
buscador_profundo = BuscadorVectorial(
    almacen, codificar_consulta, top_k=PROFUNDIDAD, ef=EF_ELEGIDO,
)

# El oraculo se construye desde el CSV -la foto ANTES de NB08-, el indice
# lleva los eventos aplicados: no contienen el mismo corpus.
ids_del_oraculo = set(ids_completo)

filas_frontera = []
for qid, vector, texto in zip(QUERY_IDS_DEV, vectores_dev, desarrollo["query_text"]):
    comun = {"qrels": QRELS[qid], "k": TOP_K, "ranking_ann": ann_dev[qid]}
    con_pre = diagnosticar_consulta(qid, ranking_oraculo=oraculo_dev[qid], **comun)
    con_post = diagnosticar_consulta(qid, ranking_oraculo=oraculo_dev_post[qid], **comun)
    if not con_pre["perdidos_por_el_ann"] and not con_post["perdidos_por_el_ann"]:
        continue
    # El ranking completo del oraculo da el puesto real, no "fuera del top-10".
    orden_oraculo = [
        r.document_id
        for r in oraculo_post.search_vector(vector, k=len(ids_post))
    ]
    puesto_oraculo = {pid: i + 1 for i, pid in enumerate(orden_oraculo)}
    orden_ann = [r.document_id for r in buscador_profundo.buscar(texto, top_k=PROFUNDIDAD)]
    # Quien ocupa el sitio: lo que el ANN trae y el oraculo PRE no puede ver.
    intrusos = [p for p in ann_dev[qid] if p not in ids_del_oraculo]
    for pid in sorted(set(con_pre["perdidos_por_el_ann"]) | set(con_post["perdidos_por_el_ann"])):
        filas_frontera.append({
            "consulta": qid,
            "producto_perdido": pid,
            "lo_pierde_vs_oraculo_pre_nb08": pid in con_pre["perdidos_por_el_ann"],
            "lo_pierde_vs_oraculo_post_nb08": pid in con_post["perdidos_por_el_ann"],
            "puesto_en_oraculo_post_nb08": puesto_oraculo.get(pid, "no esta"),
            "puesto_en_ann_ef_efectivo_200": (
                orden_ann.index(pid) + 1 if pid in orden_ann else f">{PROFUNDIDAD}"
            ),
            "altas_de_nb08_en_el_top10_del_ann": ", ".join(intrusos) or "-",
        })

# Segunda comprobacion: cuanto se separan los dos corpus, y por donde.
eventos = load_csv(DATA / "eventos_catalogo.csv")
solo_en_el_indice = sorted(
    set(eventos.loc[eventos["operation"] == "UPSERT", "product_id"]) - ids_del_oraculo
)
solo_en_el_oraculo = sorted(
    set(eventos.loc[eventos["operation"] == "DELETE", "product_id"]) & ids_del_oraculo
)
perdidos_unicos = {fila["producto_perdido"] for fila in filas_frontera}
perdidos_que_son_baja = perdidos_unicos & set(solo_en_el_oraculo)
print(f"altas de NB08 que el indice tiene y el oraculo no: {len(solo_en_el_indice)}")
print(f"   {solo_en_el_indice}")
print(f"bajas de NB08 que el oraculo aun cree vivas      : {len(solo_en_el_oraculo)}")
print(f"   {solo_en_el_oraculo}")
print(f"perdidos por el ANN que son una baja de NB08     : "
      f"{len(perdidos_que_son_baja)} de {len(perdidos_unicos)}")

# Y lo que el confound le cuesta a la fila C3 de la tabla comparativa: el
# mismo nDCG del ANN medido contra un oraculo que si ve las altas de NB08.
ndcg_pre = comparar_ndcg_con_oraculo(ann_dev, oraculo_dev, QRELS, k=TOP_K)
ndcg_post = comparar_ndcg_con_oraculo(ann_dev, oraculo_dev_post, QRELS, k=TOP_K)
tabla_dos_oraculos = pd.concat([
    ndcg_pre.assign(corpus_del_oraculo="pre-NB08 (el de la seccion B)"),
    ndcg_post.assign(corpus_del_oraculo="post-NB08 (mismo que el indice)"),
])
print()
print(tabla_dos_oraculos.to_string(index=False))
tabla_frontera = pd.DataFrame(filas_frontera)
tabla_frontera.style.hide(axis="index")

altas de NB08 que el indice tiene y el oraculo no: 8
   ['AURUM-NEW-001', 'AURUM-NEW-002', 'AURUM-NEW-003', 'AURUM-NEW-004', 'AURUM-NEW-005', 'AURUM-NEW-006', 'AURUM-NEW-007', 'AURUM-NEW-008']
bajas de NB08 que el oraculo aun cree vivas      : 8
   ['8417441271', 'B00JOH9FRO', 'B01JADTEKE', 'B07GWRF23V', 'B07RL2XB4V', 'B07S7B3SN2', 'B07XZBXCNB', 'B081JP8CC6']
perdidos por el ANN que son una baja de NB08     : 0 de 5

                        sistema  precision_at_10  recall_at_10  mrr_at_10  ndcg_at_10              corpus_del_oraculo
oráculo exacto (DenseRetriever)            0.625        0.2905     0.9375      0.6006   pre-NB08 (el de la seccion B)
              ANN elegido (R04)            0.575        0.2725     0.5625      0.4787   pre-NB08 (el de la seccion B)
oráculo exacto (DenseRetriever)            0.600        0.2829     0.6875      0.5262 post-NB08 (mismo que el indice)
              ANN elegido (R04)            0.575        0.2725     0.5625      0.4787 post-NB08 (mismo que 

consulta,producto_perdido,lo_pierde_vs_oraculo_pre_nb08,lo_pierde_vs_oraculo_post_nb08,puesto_en_oraculo_post_nb08,puesto_en_ann_ef_efectivo_200,altas_de_nb08_en_el_top10_del_ann
13357,B00YMSZDZS,True,False,11,11,AURUM-NEW-008
18868,B07GN7XRQ9,True,True,10,10,-
18868,B07H2Y8R6Y,True,True,2,2,-
18868,B07H97VGBP,True,True,1,1,-
43240,B08MQ42Z6P,True,False,11,11,AURUM-NEW-005


### G.2 · La formulación `-semantic` con menos consistencia

In [12]:
columnas_semantic = [c for c in tabla_consistencia.columns if "semantic" in c]
tabla_consistencia_ordenada = tabla_consistencia.sort_values(columnas_semantic)
peor_intencion = str(tabla_consistencia_ordenada.iloc[0]["intencion"])

print(f"intencion con menor consistencia semantic-vs-resto: {peor_intencion}")
for formulacion in ("direct", "context", "semantic"):
    eid = f"EVAL-{peor_intencion}-{formulacion}"
    texto = evaluacion.loc[evaluacion["evaluation_id"] == eid, "query_text"].iloc[0]
    print(f"  {formulacion:8s} \"{texto}\"\n"
          f"           top-5: {rankings_evaluacion[eid][:5]}")
tabla_consistencia_ordenada.style.hide(axis="index")

intencion con menor consistencia semantic-vs-resto: 93437
  direct   "sillas oficina ergonomicas"
           top-5: ['AURUM-NEW-002', 'B09DVP26N3', 'B08HGBV9R4', 'B08ZMPDXQC', 'B08GSD7FZT']
  context  "me duele la espalda al trabajar y necesito una silla con buen apoyo lumbar"
           top-5: ['AURUM-NEW-002', 'B07FPF4KHR', 'B07JGRT3KR', 'B08HGBV9R4', 'B07BGGY1HK']
  semantic "necesito un asiento cómodo para trabajar ocho horas con buen apoyo para la espalda"
           top-5: ['AURUM-NEW-002', 'B00TZYI03Q', 'B0813YXVSY', 'B00CWZRPWW', 'B01N907PBW']


intencion,jaccard_context_direct,jaccard_context_semantic,jaccard_direct_semantic
93437,0.176500,0.250000,0.052600
101352,0.538500,0.250000,0.333300
100455,0.666700,0.333300,0.250000
96202,0.333300,0.666700,0.333300


### G.2.b · PRUEBAS POST-RESULTADOS · NO DECISIONES — ¿de qué depende la robustez a la paráfrasis?

**Problema.** La intención 93437 da `jaccard_direct_semantic = 0,053`: reformular sin las palabras clave cambia casi todo el catálogo devuelto. Es capa 1 por descarte —las tres formulaciones comparten índice, `ef` y corpus—, pero "representación" tiene dos palancas: la **dimensión** (se usan 768 de las 3.072 nativas, y la robustez a paráfrasis es de lo primero que se degrada al truncar) y la **plantilla** (A4 codifica `text` comercial recortado, y R01 la eligió midiendo consultas de tipo `direct`; con paráfrasis nunca se midió).

**Se comprueba.** {A4, A3, A0} × {768, 1536, 3072} en búsqueda exacta. Cero llamadas: los vectores de las 7 plantillas y de las 12 consultas ya están en caché a 3.072 dimensiones.

⚠️ No comparar con la sección E: aquí no hay ANN ni eventos de NB08, así que `AURUM-NEW-002` no existe. `A4 · 768` es el control interno y la comparación válida es entre filas de esta tabla.

**Lectura.** `jaccard_*` va de 0 a 1, más alto es mejor. `jaccard_par_peor_de_las_4` delata si una configuración arregla la media hundiendo otra intención.

In [13]:
# PRUEBAS POST-RESULTADOS - NO DECISIONES: R01 (plantilla A4) y DIM=768 no se
# reabren. Todo sale de la cache -si algo faltara, encode_corpus pagaria
# 15.000 llamadas, asi que la celda comprueba la clave antes de entrar-.
PLANTILLAS_PRUEBA = ("A4", "A3", "A0")
DIMS_PRUEBA = (768, 1536, 3072)

for nombre in PLANTILLAS_PRUEBA:
    corpus_id = f"catalogo_productos__{nombre}"
    clave_plantilla = cache_key(
        model_id=MODELO, kind="document", contract=CONTRATO, corpus_id=corpus_id,
        fingerprint=corpus_fingerprint(render_template(completo, nombre)),
    )
    if not (CACHE / f"{clave_plantilla}.npy").exists():
        raise RuntimeError(
            f"{corpus_id} no esta en cache ({clave_plantilla}): esta celda no "
            f"paga 15.000 llamadas nuevas. Deberia estar desde NB03."
        )

vectores_eval_nativos = encode_corpus(
    _encoder, evaluacion["query_text"].tolist(), corpus_id="consultas_evaluacion",
    kind="query", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
).vectors
IDS_EVAL = evaluacion["evaluation_id"].tolist()
titulo_de = dict(zip(completo["product_id"], completo["title"]))

filas_robustez, rankings_por_config = [], {}
for nombre in PLANTILLAS_PRUEBA:
    docs_nativos = encode_corpus(
        _encoder, render_template(completo, nombre),
        corpus_id=f"catalogo_productos__{nombre}",
        kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
    ).vectors
    for dim in DIMS_PRUEBA:
        retriever = DenseRetriever(
            truncate_dim(docs_nativos, dim), ids_completo, metric="cosine",
        )
        rankings = rank_queries_dense(
            retriever, IDS_EVAL, truncate_dim(vectores_eval_nativos, dim), k=TOP_K,
        )
        tabla = formulation_consistency(rankings, k=TOP_K)
        columnas_j = [c for c in tabla.columns if c.startswith("jaccard_")]
        fila_93437 = tabla.loc[tabla["intencion"] == "93437"].iloc[0]
        rankings_por_config[(nombre, dim)] = rankings
        filas_robustez.append({
            "plantilla": nombre,
            "dim": dim,
            "jaccard_medio_4_intenciones": round(float(tabla[columnas_j].values.mean()), 4),
            "jaccard_93437_direct_vs_semantic": float(fila_93437["jaccard_direct_semantic"]),
            "jaccard_par_peor_de_las_4": round(float(tabla[columnas_j].values.min()), 4),
        })
        del retriever
    del docs_nativos

tabla_robustez = pd.DataFrame(filas_robustez).sort_values(
    "jaccard_93437_direct_vs_semantic", ascending=False
)

# Lo que la metrica no ensena: QUE productos devuelve la formulacion rota.
CONSULTA_ROTA = "EVAL-93437-semantic"
print(f'{CONSULTA_ROTA}: "'
      f'{evaluacion.loc[evaluacion["evaluation_id"] == CONSULTA_ROTA, "query_text"].iloc[0]}"')
for config in ((PLANTILLA, DIM), (PLANTILLA, 3072), ("A3", 3072), ("A0", 3072)):
    print(f"  {config[0]} · {config[1]}d")
    for pid in rankings_por_config[config][CONSULTA_ROTA][:5]:
        print(f"      {pid} · {str(titulo_de.get(pid, '?'))[:58]}")
tabla_robustez.style.hide(axis="index")

EVAL-93437-semantic: "necesito un asiento cómodo para trabajar ocho horas con buen apoyo para la espalda"
  A4 · 768d
      B08ZMPDXQC · WSDSX Office Chairs Office Chair High Back Office Desk Cha
      B09DVP26N3 · Sillas De Computadora Oficina PU Jefe Red Roja Giratoria D
      B08HGBV9R4 · PROMECITY Silla de Oficina Casera, Silla de Escritorio Erg
      846086104X · ¡Como en casa!
      B00TZYI03Q · Ayudas Dinámicas - Silla de ducha y w.c.clean, talla 49cm,
  A4 · 3072d
      B08ZMPDXQC · WSDSX Office Chairs Office Chair High Back Office Desk Cha
      B09DVP26N3 · Sillas De Computadora Oficina PU Jefe Red Roja Giratoria D
      B08HGBV9R4 · PROMECITY Silla de Oficina Casera, Silla de Escritorio Erg
      B00TZYI03Q · Ayudas Dinámicas - Silla de ducha y w.c.clean, talla 49cm,
      B00CWZRPWW · ARTICULOS SIN MARCA PREDETERM 2-0083 Rueda
  A3 · 3072d
      B08ZMPDXQC · WSDSX Office Chairs Office Chair High Back Office Desk Cha
      B092QVSY57 · Respaldo para silla de oficina, cojín e

plantilla,dim,jaccard_medio_4_intenciones,jaccard_93437_direct_vs_semantic,jaccard_par_peor_de_las_4
A3,1536,0.505400,0.818200,0.250000
A3,3072,0.512300,0.818200,0.333300
A3,768,0.454300,0.538500,0.250000
A0,3072,0.412400,0.428600,0.250000
A4,3072,0.329200,0.428600,0.176500
A0,1536,0.367300,0.428600,0.250000
A4,768,0.374700,0.333300,0.250000
A4,1536,0.341800,0.333300,0.176500
A0,768,0.372700,0.250000,0.250000


### G.3 · Conclusión

#### Antes de atribuir: una corrección de la propia medición

Al verificar los fallos se detectó que **el oráculo y el índice no contenían el mismo catálogo**. NB08 aplicó 24 eventos sobre Qdrant (8 bajas, 8 actualizaciones, 8 altas), pero el oráculo se seguía construyendo desde `catalogo_productos.csv`, la foto anterior. Ambos suman 15.000 puntos y no son los mismos. Como varias altas responden literalmente a consultas de desarrollo —`AURUM-NEW-008` se titula *"Base tapizada 160 x 200 sin patas"*, que es la consulta 13357—, el ANN las devolvía y `perdidos_por_el_ann` lo apuntaba como pérdida suya.

No es una decisión de diseño ni de negocio: es un **defecto de la medición**, detectado al comprobar por qué subir `ef` no recuperaba nada. Se recalculó el oráculo sobre el corpus post-NB08 para equipararlo al índice (G.1.c) y se repitió la comparación, que lo corroboró: la brecha oráculo→ANN cae de **0,1219 a 0,0475 de nDCG@10**. El 61 % de la pérdida atribuida al ANN no existía.

La cifra corregida se valida sola: **0,0475 es exactamente la brecha que midió NB06** (`config.yaml` → `nb06_ann.r04_ef_elegido.ndcg_vs_oraculo`: 60,06 % del oráculo frente a 55,31 % de R04, −4,75 puntos), cuando aún no existía ninguna mutación. Las altas de NB08 hunden por igual al oráculo y al ANN —no tienen juicio y por D04 puntúan 0—, así que la brecha se conserva. Dos mediciones independientes, separadas por tres notebooks y 24 eventos de escritura, dan el mismo 4,75: eso es lo que confirma que 0,1219 era el artefacto.

#### Los tres fallos, uno por capa

| Consulta | Capa | Motivo | Evidencia |
|---|---|---|---|
| **18868** "botines marrones mujer tacon medio" | **Índice** | densidad de la región (causa ya establecida en NB06): explorando solo 32 candidatos, el recorrido HNSW los consume en vecinos próximos pero no óptimos | `B07H97VGBP` es el **vecino nº 1** del oráculo corregido y no aparece en el top-10 con `ef=32` ni `ef=64` (0 de 3 relevantes). Con `ef=128` vuelven los tres (3 de 3) |
| **93437** "sillas oficina ergonómicas" | **Representación** | pesa la **plantilla**, no la dimensión: A4 codifica el `text` comercial, saturado de palabras clave | Jaccard `direct`-vs-`semantic`: **0,333** (A4·768) → **0,818** (A3·3072). Con A4 el top-5 de la formulación parafraseada trae una silla de ducha y *"¡Como en casa!"*; con A3·3072 los cinco son sillas de oficina |
| **33633** "disfraz halloween talla grande hombre" | **Datos** | *pooling bias*: el juicio no cubre el catálogo | 22 productos con "disfraz"+"hombre" en el catálogo, **0** en el pool de 16 juzgados. El único `E` es una blusa de mujer (`B07GSVQG2R`) |

#### Descartadas tras la corrección: 13357 y 43240

El ANN sitúa `B00YMSZDZS` y `B08MQ42Z6P` en el **puesto 11**, exactamente donde los pone el oráculo corregido, y los pierde igual con `ef` 32, 64, 128 y 256 — si fuera aproximación, `ef=256` los recuperaría. Lo que ocupó su asiento fue un producto insertado por NB08 (`AURUM-NEW-008` y `AURUM-NEW-005`). No son fallos de ninguna capa.

#### Mejora medida, no aplicada

`ef=128` recupera los 3 relevantes de la 18868 y su p95 —**12,36 ms** según `benchmark_ann.csv`— cabe dentro del presupuesto de 20 ms que fijó D16. **R04 se mantiene en `ef=32`**: la decisión se tomó antes de ver la curva y así se deja.

Conviene precisar el crédito: **NB06 ya sabía esto**. `nb06_ann.r04_ef_elegido.caso_duro_18868` documenta que esta consulta tiene recall 0 con `ef=32` y se recupera entera con `ef=128`, y R04 eligió `ef=32` sabiéndolo, por ser la de menor p95 entre las cuatro que cumplían D16. Lo que aporta NB09 es repetir el barrido sobre el **índice ya mutado** —el estado real del sistema entregado— y confirmar que el diagnóstico se mantiene tras los 24 eventos. Lo que queda anotado para una iteración futura no es "subir `ef`" sin más, sino **cambiar el criterio de desempate** de R04: menor p95 entre las admisibles premia a una configuración que sacrifica una consulta entera; un criterio que mirase el recall mínimo por consulta, y no solo el agregado, habría elegido `ef=128` sin salirse de D16.

Lo mismo aplica a A3 frente a A4: R01 eligió midiendo nDCG@10 sobre consultas de tipo `direct`, y la robustez a la paráfrasis no entró en esa medición.

#### Defecto colateral detectado

NB08 codificó los 16 `UPSERT` con `text` **crudo**, sin pasar por la plantilla A4. Irrelevante para las 8 altas (66-96 caracteres, por debajo del corte de 936), pero **4 de las 8 actualizaciones lo superan** —hasta 2.676— y quedaron indexadas sin recortar frente a los otros 14.996 puntos de la colección.

## H · Los artefactos

In [14]:
destino_tabla = Path("..") / "artifacts" / "tabla_comparativa.md"

# La brecha C2->C3 leida en crudo sobrevalora la perdida del ANN: el oraculo
# de C2 no comparte corpus con el indice que mide C3 (G.1.c). La tabla no se
# toca -son las cifras que midio cada notebook-, se le anade el contexto.
if "tabla_dos_oraculos" not in globals():
    raise RuntimeError(
        "Ejecuta G.1.c antes que H: la nota al pie de C3 sale de ahi."
    )


def _ndcg_de(sistema, corpus):
    fila = tabla_dos_oraculos[
        (tabla_dos_oraculos["sistema"] == sistema)
        & (tabla_dos_oraculos["corpus_del_oraculo"].str.startswith(corpus))
    ]
    return float(fila.iloc[0]["ndcg_at_10"])


BRECHA_PRE = _ndcg_de("oráculo exacto (DenseRetriever)", "pre") - _ndcg_de(
    "ANN elegido (R04)", "pre"
)
BRECHA_POST = _ndcg_de("oráculo exacto (DenseRetriever)", "post") - _ndcg_de(
    "ANN elegido (R04)", "post"
)
NOTA_C3 = (
    "\n\n> **Nota sobre la fila C3.** La distancia C2→C3 leída en crudo "
    f"({BRECHA_PRE:.4f} de nDCG@10) **sobrevalora la pérdida del ANN**. El "
    "oráculo de C2 se construye sobre `catalogo_productos.csv` —el estado "
    "previo a NB08—, mientras que el índice que mide C3 lleva aplicados los 24 "
    "eventos del ciclo de vida: 8 bajas y 8 altas de diferencia. Varias de esas "
    "altas responden literalmente a consultas de desarrollo (`AURUM-NEW-008` = "
    "\"Base tapizada 160 x 200 sin patas\" para la consulta 13357), así que el "
    "ANN las devuelve y no recibe crédito por ellas —no tienen juicio en los "
    "qrels, y por D04 puntúan 0— mientras desplazan a relevantes juzgados. "
    f"Recalculado el oráculo sobre el mismo corpus del índice, la brecha real "
    f"es **{BRECHA_POST:.4f}**: el {1 - BRECHA_POST / BRECHA_PRE:.0%} de la "
    "pérdida aparente era diferencia de corpus, no aproximación ANN. "
    "Evidencia en las secciones G.1.c y G.3."
)
destino_tabla.write_text(
    tabla_comparativa_final.to_markdown(index=False) + NOTA_C3, encoding="utf-8"
)

print(f"Escrito {destino_tabla} (con nota al pie de C3: brecha "
      f"{BRECHA_PRE:.4f} -> {BRECHA_POST:.4f})")
print(f"Escrito {destino_busqueda} · {len(resultados_busqueda)} filas (seccion D)")
print(f"Consistencia entre formulaciones: {len(tabla_consistencia)} intenciones (seccion E)")
print(f"Consultas filtradas puras: {FILTROS_OK}/{len(tabla_filtros)} (seccion F)")

Escrito ..\artifacts\tabla_comparativa.md (con nota al pie de C3: brecha 0.1219 -> 0.0475)
Escrito ..\resultados\resultados_busqueda.csv · 120 filas (seccion D)
Consistencia entre formulaciones: 4 intenciones (seccion E)
Consultas filtradas puras: 4/4 (seccion F)
